# 🌐 Notebook 2 — MarianMT Fine-tuning (Baseline)
**Project:** Uzbek NMT | **Author:** Asliddin

---
Fine-tune **Helsinki-NLP MarianMT** models as a lightweight baseline.
One model per direction — fast, low memory, good starting point.

**Expected:** BLEU ~24-28 across directions

In [ ]:
import sys
sys.path.append('../src')
import torch
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup
from tqdm import tqdm
import csv, time
from pathlib import Path

from dataset import get_dataloaders
from model import build_marian, save_model, translate_marian
from evaluate import compute_bleu_chrf, plot_training_curves

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Train MarianMT for each direction

In [ ]:
DIRECTIONS = ['uz-en','en-uz','uz-ru','ru-uz']
EPOCHS = 3
LR = 5e-5
marian_results = {}

for direction in DIRECTIONS:
    print(f'\n=== Training {direction} ===')
    model, tokenizer = build_marian(direction)
    if model is None:
        print(f'Skipping {direction} — no pre-trained MarianMT available')
        continue
    model = model.to(device)

    try:
        train_loader, val_loader, test_loader = get_dataloaders(
            direction=direction, tokenizer=tokenizer,
            data_dir='../data/processed', batch_size=32, model_type='marian')
    except Exception as e:
        print(f'Data not ready for {direction}: {e}')
        continue

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(len(train_loader)*EPOCHS*0.1), len(train_loader)*EPOCHS)

    best_bleu = 0
    for epoch in range(1, EPOCHS+1):
        model.train()
        train_loss = 0
        for batch in tqdm(train_loader, desc=f'Ep{epoch} Train', leave=False):
            batch = {k:v.to(device) for k,v in batch.items()}
            optimizer.zero_grad()
            loss = model(**batch).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            train_loss += loss.item()

        model.eval()
        preds, refs = [], []
        with torch.no_grad():
            for batch in val_loader:
                batch = {k:v.to(device) for k,v in batch.items()}
                gen = model.generate(batch['input_ids'], attention_mask=batch['attention_mask'],
                                     num_beams=4, max_length=128)
                p = tokenizer.batch_decode(gen, skip_special_tokens=True)
                labels = batch['labels'].clone()
                labels[labels==-100] = tokenizer.pad_token_id
                r = tokenizer.batch_decode(labels, skip_special_tokens=True)
                preds.extend(p); refs.extend(r)

        bleu, chrf = compute_bleu_chrf(refs, preds)
        print(f'  Ep {epoch} | BLEU: {bleu:.2f} | chrF: {chrf:.4f}')

        if bleu > best_bleu:
            best_bleu = bleu
            save_model(model, tokenizer, f'../models/marian/{direction}')

    marian_results[direction] = {'bleu': best_bleu}
    print(f'Best BLEU for {direction}: {best_bleu:.2f}')

print('\nMarianMT training complete!')
print(marian_results)